In [1]:
# ============================================================
# CodPerformancePanel.ipynb
# ============================================================
# Script AUTONOMO — non modifica CodMatchFreddieHMDA.
#
# INPUT:
#   1. matched_{YEAR}.csv  output di CodMatchFreddieHMDA
#      (una riga per loan, con dati origination Freddie + HMDA)
#   2. Stessi zip Freddie Mac (stessa struttura del notebook originale)
#      I file TIME (performance) sono estratti dagli stessi zip.
#
# OUTPUT:
#   panel_{YEAR}.csv
#   Una riga per (loan_sequence_number, quarter)
#
#   Struttura colonne:
#     [A] Identificatori     : loan_sequence_number, period_quarter, ...
#     [B] Performance        : current_upb, delinq, loan_age, ltv, ...
#     [C] Target survival    : default, prepayment
#     [D] Origination + HMDA : tutte le colonne del matched (time-invariant)
#
# STRUTTURA DRIVE ATTESA:
#   MyDrive/
#     thesis_data/
#       freddie/
#         historical_data_{YEAR}/
#           historical_data_{YEAR}Q1.zip  <- contiene anche i file time
#           historical_data_{YEAR}Q2.zip
#           ...
#       output/
#         matched_{YEAR}.csv              <- input
#         panel_{YEAR}.csv               <- output
# ============================================================

!pip install pyarrow fastparquet -q

from google.colab import drive
drive.mount('/content/drive')

import os
print('Drive montato:', os.path.exists('/content/drive/MyDrive'))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.5 MB/s eta 0:00:00
Mounted at /content/drive
Drive montato: True


In [2]:
# ============================================================
# CELLA 1 - Configurazione
# ============================================================
# MODIFICA SOLO DRIVE_ROOT e YEAR.

import os, glob, zipfile, shutil, gc
import pandas as pd
import numpy as np

# ---- UNICI DUE VALORI DA MODIFICARE ----
DRIVE_ROOT = '/content/drive/MyDrive/thesis_data'
YEAR       = 2024
# ----------------------------------------

FREDDIE_DIR  = os.path.join(DRIVE_ROOT, 'freddie')
OUTPUT_DIR   = os.path.join(DRIVE_ROOT, 'output')
PERF_LOCAL   = '/content/freddie_perf_local'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PERF_LOCAL, exist_ok=True)

MATCHED_PATH = os.path.join(OUTPUT_DIR, f'matched_{YEAR}.csv')
OUTPUT_PATH  = os.path.join(OUTPUT_DIR, f'panel_{YEAR}.csv')

print(f'Anno             : {YEAR}')
print(f'Freddie zip dir  : {FREDDIE_DIR}')
print(f'File matched     : {MATCHED_PATH}')
print(f'Output panel     : {OUTPUT_PATH}')
print()
for label, path in [('Freddie zip dir', FREDDIE_DIR), ('matched CSV', MATCHED_PATH)]:
    status = 'OK' if os.path.exists(path) else 'ERRORE - non trovato'
    print(f'  {status}  {label}: {path}')

Anno             : 2024
Freddie zip dir  : /content/drive/MyDrive/thesis_data/freddie
File matched     : /content/drive/MyDrive/thesis_data/output/matched_2024.csv
Output panel     : /content/drive/MyDrive/thesis_data/output/panel_2024.csv

  OK  Freddie zip dir: /content/drive/MyDrive/thesis_data/freddie
  OK  matched CSV: /content/drive/MyDrive/thesis_data/output/matched_2024.csv


In [21]:
# ============================================================
# CELLA 2 - Layout colonne file performance
# ============================================================
# Manuale ufficiale Freddie Mac Single Family Loan-Level Dataset.
# Gestiamo automaticamente versioni con meno colonne (dataset piu' vecchi).

PERF_COLS_FULL = [
    'loan_sequence_number',
    'monthly_reporting_period',
    'current_upb',
    'current_loan_delinquency_status',
    'loan_age',
    'remaining_months_to_maturity',
    'defect_settlement_date',
    'modifications_flag',
    'zero_balance_code',
    'zero_balance_effective_date',
    'current_interest_rate',
    'current_deferred_upb',
    'due_date_last_paid_installment',
    'mi_recoveries',
    'net_sales_proceeds',
    'non_mi_recoveries',
    'expenses',
    'legal_costs',
    'maintenance_costs',
    'taxes_and_insurance',
    'miscellaneous_expenses',
    'actual_loss',
    'modification_cost',
    'step_modification_flag',
    'deferred_payment_plan',
    'estimated_ltv',
    'zero_balance_removal_upb',
    'delinquent_accrued_interest',
    'delinquency_due_to_disaster',
    'borrower_assistance_status',
    'current_month_modification_loss',
    'interest_bearing_upb',
]
# Colonne da tenere nell'output (le colonne finanziarie di recovery
# come legal_costs, taxes, ecc. sono escluse per default;
# aggiungile qui se le vuoi nel panel)
PERF_COLS_KEEP = [
    'loan_sequence_number',
    'monthly_reporting_period',
    'current_upb',
    'current_loan_delinquency_status',
    'loan_age',
    'remaining_months_to_maturity',
    'modifications_flag',
    'zero_balance_code',
    'zero_balance_effective_date',
    'current_interest_rate',
    'current_deferred_upb',
    'estimated_ltv',
    'delinquency_due_to_disaster',
    'borrower_assistance_status',
]

print(f'Colonne nel manuale  : {len(PERF_COLS_FULL)}')
print(f'Colonne da mantenere : {len(PERF_COLS_KEEP)}')

Colonne nel manuale  : 32
Colonne da mantenere : 14


In [22]:
# ============================================================
# CELLA 3 - Estrazione file TIME dagli zip Freddie
# ============================================================
# I file performance (es. time_2024Q1.txt) sono dentro gli stessi zip
# dell'origination ma con 'time' nel nome.
# La Cella 2b di CodMatchFreddieHMDA li ignorava; qui li estraiamo.

def extract_performance_zips(year):
    zip_pattern_sub  = os.path.join(FREDDIE_DIR, f'historical_data_{year}',
                                    f'historical_data_{year}Q*.zip')
    zip_pattern_flat = os.path.join(FREDDIE_DIR, f'historical_data_{year}Q*.zip')
    zip_files = glob.glob(zip_pattern_sub) or glob.glob(zip_pattern_flat)

    if not zip_files:
        raise FileNotFoundError(
            f'Nessun zip trovato per {year}.\n'
            f'Pattern cercati:\n  {zip_pattern_sub}\n  {zip_pattern_flat}'
        )

    print(f'Trovati {len(zip_files)} zip per {year}:')
    extracted = []

    for zpath in sorted(zip_files):
        zname = os.path.basename(zpath)
        with zipfile.ZipFile(zpath, 'r') as z:
            contents = z.namelist()
            time_files = [f for f in contents
                          if 'time' in f.lower() and f.endswith('.txt')]

            if not time_files:
                print(f'  ATTENZIONE: nessun file TIME in {zname}')
                print(f'  Contenuto zip: {contents}')
                continue

            for tf in time_files:
                out_name = os.path.basename(tf)
                dest = os.path.join(PERF_LOCAL, out_name)

                if os.path.exists(dest):
                    size_mb = os.path.getsize(dest) / 1e6
                    print(f'  {zname} -> {out_name} gia estratto ({size_mb:.0f} MB), skip')
                else:
                    print(f'  Estraggo {out_name} da {zname}...', end=' ', flush=True)
                    with z.open(tf) as src, open(dest, 'wb') as dst:
                        shutil.copyfileobj(src, dst)
                    size_mb = os.path.getsize(dest) / 1e6
                    print(f'OK ({size_mb:.0f} MB)')

                extracted.append(dest)

    if not extracted:
        raise RuntimeError(
            'Nessun file TIME estratto. Verifica che gli zip contengano '
            'file con "time" nel nome (es. time_2024Q1.txt).'
        )
    return sorted(extracted)


print(f'Estraggo file performance per {YEAR}...')
perf_txt_files = extract_performance_zips(YEAR)
print(f'\nFile TIME estratti: {len(perf_txt_files)}')
for f in perf_txt_files:
    print(f'  {os.path.basename(f)}  ({os.path.getsize(f)/1e6:.0f} MB)')

Estraggo file performance per 2024...
Trovati 4 zip per 2024:
  historical_data_2024Q1.zip -> historical_data_time_2024Q1.txt gia estratto (330 MB), skip
  historical_data_2024Q2.zip -> historical_data_time_2024Q2.txt gia estratto (334 MB), skip
  historical_data_2024Q3.zip -> historical_data_time_2024Q3.txt gia estratto (321 MB), skip
  historical_data_2024Q4.zip -> historical_data_time_2024Q4.txt gia estratto (240 MB), skip

File TIME estratti: 4
  historical_data_time_2024Q1.txt  (330 MB)
  historical_data_time_2024Q2.txt  (334 MB)
  historical_data_time_2024Q3.txt  (321 MB)
  historical_data_time_2024Q4.txt  (240 MB)


In [23]:
with open(perf_txt_files[0], 'r') as fh:
    for i, line in enumerate(fh):
        if i >= 3: break
        print(repr(line))

'F24Q10000001|202402|82000.00|0|000|360|||||8.000|0.00||||||||||||||999||||||82000.00\n'
'F24Q10000001|202403|82000.00|0|001|359|||||8.000|0.00||||||||||||||999||||||82000.00\n'
'F24Q10000001|202404|82000.00|0|002|358|||||8.000|0.00||||||||||||||999||||||82000.00\n'


In [24]:
# ============================================================
# CELLA 4 - Carica matched e performance
# ============================================================

# --- 4a. Carica il matched (una riga per loan) ---
print(f'Carico matched: {MATCHED_PATH}')
matched = pd.read_csv(MATCHED_PATH, dtype=str, low_memory=False)
print(f'  {len(matched):,} loan x {len(matched.columns)} colonne')

loan_ids = set(matched['loan_sequence_number'].dropna().unique())
print(f'  Loan ID unici: {len(loan_ids):,}')

# --- 4b. Carica i file performance, filtrando solo i loan del matched ---
print(f'\nCarico file performance...')
perf_frames = []

for f in perf_txt_files:
    fname = os.path.basename(f)

    # Rileva numero colonne dalla prima riga per adattare il layout
    with open(f, 'r') as fh:
        first_line = fh.readline()
    n_cols_file = len(first_line.split('|'))
    col_names = PERF_COLS_FULL[:n_cols_file]

    df = pd.read_csv(
    f, sep='|', header=None,
    names=PERF_COLS_FULL[:n_cols_file],
    dtype=str, low_memory=False
    )


    # Filtra subito per ridurre la RAM
    df = df[df['loan_sequence_number'].isin(loan_ids)]

    # Tieni solo le colonne di interesse
    cols_available = [c for c in PERF_COLS_KEEP if c in df.columns]
    df = df[cols_available]

    print(f'  {fname}: {len(df):,} righe (colonne nel file: {n_cols_file})')
    perf_frames.append(df)

perf = pd.concat(perf_frames, ignore_index=True)
del perf_frames
gc.collect()


print(f'\nPerformance totale  : {len(perf):,} righe')
print(f'Loan unici in perf  : {perf["loan_sequence_number"].nunique():,}')

Carico matched: /content/drive/MyDrive/thesis_data/output/matched_2024.csv
  41,504 loan x 97 colonne
  Loan ID unici: 41,504

Carico file performance...
Prime 2 righe:
  loan_sequence_number monthly_reporting_period current_upb current_loan_delinquency_status loan_age remaining_months_to_maturity defect_settlement_date modifications_flag zero_balance_code zero_balance_effective_date current_interest_rate current_deferred_upb due_date_last_paid_installment mi_recoveries net_sales_proceeds non_mi_recoveries expenses legal_costs maintenance_costs taxes_and_insurance miscellaneous_expenses actual_loss modification_cost step_modification_flag deferred_payment_plan estimated_ltv zero_balance_removal_upb delinquent_accrued_interest delinquency_due_to_disaster borrower_assistance_status current_month_modification_loss interest_bearing_upb
0         F24Q10000001                   202402    82000.00                               0      000                          360                    NaN    

In [25]:
# ============================================================
# CELLA 5 - Preparazione variabili performance e target
# ============================================================

# Conversioni numeriche
for col in ['current_upb', 'loan_age', 'remaining_months_to_maturity',
            'current_interest_rate', 'estimated_ltv', 'current_deferred_upb']:
    if col in perf.columns:
        perf[col] = pd.to_numeric(perf[col], errors='coerce')

# Parsing del periodo YYYYMM
perf['monthly_reporting_period'] = perf['monthly_reporting_period'].str.strip()
perf['period_year']  = perf['monthly_reporting_period'].str[:4].astype(int)
perf['period_month'] = perf['monthly_reporting_period'].str[4:6].astype(int)
perf['quarter']      = ((perf['period_month'] - 1) // 3 + 1).astype('Int64')
perf['period_quarter'] = perf['period_year'].astype(str) + 'Q' + perf['quarter'].astype(str)

# Delinquency status numerico
# 0=corrente, 1-12=mesi di ritardo, XX=12+, R/RA=REO
perf['delinq_numeric'] = pd.to_numeric(
    perf['current_loan_delinquency_status']
      .str.strip()
      .replace({'XX': '12', 'R': '13', 'RA': '13'}),
    errors='coerce'
)

# Target: default = 1 se delinquency >= 90 gg (3 mesi) oppure zero_balance REO/terza parte
default_by_delinq = perf['delinq_numeric'] >= 3
if 'zero_balance_code' in perf.columns:
    default_by_zbc   = perf['zero_balance_code'].str.strip().isin(['02', '03'])
    perf['default']  = (default_by_delinq | default_by_zbc).astype(int)
else:
    perf['default']  = default_by_delinq.astype(int)

# Target: prepayment (per competing risks)
if 'zero_balance_code' in perf.columns:
    perf['prepayment'] = perf['zero_balance_code'].str.strip().isin(['01']).astype(int)

print('Variabili performance pronte.')
print(f'  Osservazioni mensili : {len(perf):,}')
print(f'  Default events       : {perf["default"].sum():,}')
if 'prepayment' in perf.columns:
    print(f'  Prepayment events    : {perf["prepayment"].sum():,}')
print(f'  Range periodi        : {perf["period_quarter"].min()} - {perf["period_quarter"].max()}')

Variabili performance pronte.
  Osservazioni mensili : 564,079
  Default events       : 748
  Prepayment events    : 4,213
  Range periodi        : 2024Q1 - 2025Q3


In [26]:
# ============================================================
# CELLA 6 - I dati performance sono mensili, nessun collasso
# ============================================================
# Aggiungiamo solo le colonne temporali utili.

perf['period_year']  = perf['monthly_reporting_period'].str[:4].astype(int)
perf['period_month'] = perf['monthly_reporting_period'].str[4:6].astype(int)
perf['quarter']      = ((perf['period_month'] - 1) // 3 + 1).astype(int)
perf['period_quarter'] = perf['period_year'].astype(str) + 'Q' + perf['quarter'].astype(str)

perf_q = perf  # nessun collasso, una riga per (loan, mese)

print(f'Osservazioni mensili : {len(perf_q):,}')
print(f'Loan unici           : {perf_q["loan_sequence_number"].nunique():,}')
print(f'Mesi medi per loan   : {len(perf_q) / perf_q["loan_sequence_number"].nunique():.1f}')

Osservazioni mensili : 564,079
Loan unici           : 41,504
Mesi medi per loan   : 13.6


In [27]:
# ============================================================
# CELLA 7 - Join con matched (origination + HMDA)
# ============================================================
# Il matched ha una riga per loan -> viene replicata per ogni trimestre.

print('Join performance x origination/HMDA...')

panel = perf_q.merge(
    matched,
    on='loan_sequence_number',
    how='inner'   # solo loan presenti in entrambi
)

gc.collect()

print(f'Panel: {panel.shape[0]:,} righe x {panel.shape[1]} colonne')
print(f'Loan unici nel panel: {panel["loan_sequence_number"].nunique():,}')

# Verifica duplicati su (loan, quarter)
dups = panel.duplicated(subset=['loan_sequence_number', 'period_quarter']).sum()
if dups > 0:
    print(f'ATTENZIONE: {dups:,} righe duplicate su (loan, quarter).')
else:
    print('OK: nessun duplicato su (loan_sequence_number, period_quarter).')

Join performance x origination/HMDA...
Panel: 564,079 righe x 117 colonne
Loan unici nel panel: 41,504
ATTENZIONE: 360,585 righe duplicate su (loan, quarter).


In [28]:
# ============================================================
# CELLA 8 - Ordinamento colonne e salvataggio
# ============================================================

# [A] Identificatori e chiavi temporali
id_cols = [
    'loan_sequence_number',
    'period_quarter',
    'period_year',
    'quarter_number',
    'monthly_reporting_period',
]

# [B] Variabili di performance (time-varying)
perf_out_cols = [
    'loan_age',
    'remaining_months_to_maturity',
    'current_upb',
    'current_interest_rate',
    'estimated_ltv',
    'current_deferred_upb',
    'current_loan_delinquency_status',
    'delinq_numeric',
    'modifications_flag',
    'zero_balance_code',
    'zero_balance_effective_date',
    'delinquency_due_to_disaster',
    'borrower_assistance_status',
]

# [C] Target survival model
target_cols = [c for c in ['default', 'prepayment'] if c in panel.columns]

# [D] Origination + HMDA (tutte le colonne del matched, time-invariant)
already_placed = set(id_cols + perf_out_cols + target_cols)
origination_cols = [c for c in matched.columns
                    if c != 'loan_sequence_number' and c not in already_placed]

# Costruisci ordine finale
ordered = []
seen = set()
for group in [id_cols, perf_out_cols, target_cols, origination_cols]:
    for c in group:
        if c in panel.columns and c not in seen:
            ordered.append(c)
            seen.add(c)

remaining = [c for c in panel.columns if c not in seen]
ordered += remaining

panel = panel[ordered]
panel.sort_values(['loan_sequence_number', 'period_quarter'], inplace=True)
panel.reset_index(drop=True, inplace=True)

# Salva
panel.to_csv(OUTPUT_PATH, index=False)
size_mb = os.path.getsize(OUTPUT_PATH) / 1e6

print(f'Salvato: {OUTPUT_PATH}')
print(f'Dimensione   : {size_mb:.1f} MB')
print(f'Righe        : {len(panel):,}')
print(f'Colonne      : {len(panel.columns)}')
print()
print('--- COLONNE NEL PANEL ---')
print(f'[A] Identificatori   : {[c for c in id_cols       if c in panel.columns]}')
print(f'[B] Performance      : {[c for c in perf_out_cols if c in panel.columns]}')
print(f'[C] Target survival  : {target_cols}')
print(f'[D] Origination/HMDA : {len([c for c in origination_cols if c in panel.columns])} colonne')
if remaining:
    print(f'[E] Altre            : {remaining}')
print()
panel.head(3)

Salvato: /content/drive/MyDrive/thesis_data/output/panel_2024.csv
Dimensione   : 301.6 MB
Righe        : 564,079
Colonne      : 117

--- COLONNE NEL PANEL ---
[A] Identificatori   : ['loan_sequence_number', 'period_quarter', 'period_year', 'monthly_reporting_period']
[B] Performance      : ['loan_age', 'remaining_months_to_maturity', 'current_upb', 'current_interest_rate', 'estimated_ltv', 'current_deferred_upb', 'current_loan_delinquency_status', 'delinq_numeric', 'modifications_flag', 'zero_balance_code', 'zero_balance_effective_date', 'delinquency_due_to_disaster', 'borrower_assistance_status']
[C] Target survival  : ['default', 'prepayment']
[D] Origination/HMDA : 96 colonne
[E] Altre            : ['period_month', 'quarter']



,loan_sequence_number,period_quarter,period_year,monthly_reporting_period,loan_age,remaining_months_to_maturity,current_upb,current_interest_rate,estimated_ltv,current_deferred_upb,...,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units,occupancy_status,period_month,quarter
0,F24Q10000008,2024Q1,2024,202402,0,360,335000.0,8.0,84,0.0,...,3128,11.76,111800,113.0,599,857,33,P,2,1
1,F24Q10000008,2024Q1,2024,202403,1,359,335000.0,8.0,84,0.0,...,3128,11.76,111800,113.0,599,857,33,P,3,1
2,F24Q10000008,2024Q2,2024,202404,2,358,335000.0,8.0,81,0.0,...,3128,11.76,111800,113.0,599,857,33,P,4,2
